In [3]:
# ============================================================
# LOAD DATA FOR CATBOOST EXPERIMENT
# ============================================================

import pandas as pd

# Load cleaned dataset
df = pd.read_csv("../data/diamonds (cleaned).csv")

print("Dataset Shape:", df.shape)

# Target
target = "Price"

# Features
features = [col for col in df.columns if col != target]

X = df[features].copy()
y = df[target].copy()

print("X Shape:", X.shape)
print("y Shape:", y.shape)

print("\nFeature columns:")
print(features)

print("\nTarget:")
print(target)

Dataset Shape: (6485, 18)
X Shape: (6485, 17)
y Shape: (6485,)

Feature columns:
['Shape', 'Cut', 'Color', 'Clarity', 'Carat Weight', 'Length/Width Ratio', 'Depth %', 'Table %', 'Polish', 'Symmetry', 'Girdle', 'Culet', 'Length', 'Width', 'Height', 'Type', 'Fluorescence']

Target:
Price


In [4]:
# ============================================================
# CATBOOST REGRESSOR — BASELINE EXPERIMENT
# ============================================================

from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Prepare data
# ------------------------------------------------------------

X_cat = X.copy()
y_cat = y.copy()

# CatBoost needs categorical columns as strings
categorical_cols = X_cat.select_dtypes(include=["object", "category"]).columns.tolist()

for col in categorical_cols:
    X_cat[col] = X_cat[col].fillna("Unknown").astype(str)

# Numerical missing values
numerical_cols = X_cat.select_dtypes(exclude=["object", "category"]).columns.tolist()

for col in numerical_cols:
    X_cat[col] = X_cat[col].fillna(X_cat[col].median())

# ------------------------------------------------------------
# Train-test split
# ------------------------------------------------------------

from sklearn.model_selection import train_test_split

X_train_cat, X_test_cat, y_train_cat, y_test_cat = train_test_split(
    X_cat,
    y_cat,
    test_size=0.20,
    random_state=42
)

cat_features_indices = [
    X_cat.columns.get_loc(col)
    for col in categorical_cols
]

# ------------------------------------------------------------
# CatBoost Model
# ------------------------------------------------------------

cat_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.03,
    depth=6,
    loss_function="RMSE",
    random_seed=42,
    verbose=False
)

cat_model.fit(
    X_train_cat,
    y_train_cat,
    cat_features=cat_features_indices
)

# ------------------------------------------------------------
# Prediction
# ------------------------------------------------------------

cat_pred = cat_model.predict(X_test_cat)

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

cat_mae = mean_absolute_error(y_test_cat, cat_pred)
cat_rmse = np.sqrt(mean_squared_error(y_test_cat, cat_pred))
cat_r2 = r2_score(y_test_cat, cat_pred)

print("=" * 55)
print("CATBOOST REGRESSOR RESULTS")
print("=" * 55)
print(f"MAE  : {cat_mae:.4f}")
print(f"RMSE : {cat_rmse:.4f}")
print(f"R²   : {cat_r2:.4f}")
print("=" * 55)

print("\nCurrent Best Tuned Random Forest:")
print(f"MAE  : {tuned_mae:.4f}")
print(f"RMSE : {tuned_rmse:.4f}")
print(f"R²   : {tuned_r2:.4f}")

print("\nCatBoost vs Random Forest:")
print(f"R² Difference : {cat_r2 - tuned_r2:+.4f}")
print(f"MAE Difference: {cat_mae - tuned_mae:+.4f}")
print(f"RMSE Difference: {cat_rmse - tuned_rmse:+.4f}")

C:\Users\ishan\AppData\Local\Temp\ipykernel_2364\617334156.py:18: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_cat.select_dtypes(include=["object", "category"]).columns.tolist()


CATBOOST REGRESSOR RESULTS
MAE  : 432.3325
RMSE : 1242.6334
R²   : 0.8549

Current Best Tuned Random Forest:


NameError: name 'tuned_mae' is not defined

In [5]:
# ============================================================
# FAIR CATBOOST COMPARISON
# ============================================================

from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Fresh copy
X_cb = X.copy()
y_cb = y.copy()

# Split FIRST
X_train_cb, X_test_cb, y_train_cb, y_test_cb = train_test_split(
    X_cb,
    y_cb,
    test_size=0.20,
    random_state=42
)

# Identify categorical columns
cat_cols_cb = X_train_cb.select_dtypes(
    include=["object", "category"]
).columns.tolist()

# CatBoost categorical values must not contain NaN
for col in cat_cols_cb:
    X_train_cb[col] = X_train_cb[col].fillna("Unknown").astype(str)
    X_test_cb[col] = X_test_cb[col].fillna("Unknown").astype(str)

cat_indices_cb = [
    X_train_cb.columns.get_loc(col)
    for col in cat_cols_cb
]

# ------------------------------------------------------------
# CatBoost
# ------------------------------------------------------------

fair_cat_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.03,
    depth=6,
    loss_function="RMSE",
    random_seed=42,
    verbose=False
)

fair_cat_model.fit(
    X_train_cb,
    y_train_cb,
    cat_features=cat_indices_cb
)

# Prediction
fair_cat_pred = fair_cat_model.predict(X_test_cb)

# Evaluation
fair_cat_mae = mean_absolute_error(y_test_cb, fair_cat_pred)
fair_cat_rmse = np.sqrt(
    mean_squared_error(y_test_cb, fair_cat_pred)
)
fair_cat_r2 = r2_score(y_test_cb, fair_cat_pred)

print("=" * 60)
print("FAIR CATBOOST RESULTS")
print("=" * 60)
print(f"MAE  : {fair_cat_mae:.4f}")
print(f"RMSE : {fair_cat_rmse:.4f}")
print(f"R²   : {fair_cat_r2:.4f}")
print("=" * 60)

# Known previous best result
print("\nPREVIOUS BEST TUNED RANDOM FOREST")
print("=" * 60)
print("MAE  : 386.8777")
print("RMSE : 1097.1843")
print("R²   : 0.8820")


C:\Users\ishan\AppData\Local\Temp\ipykernel_2364\652169171.py:23: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols_cb = X_train_cb.select_dtypes(


FAIR CATBOOST RESULTS
MAE  : 435.0702
RMSE : 1286.4037
R²   : 0.8445

PREVIOUS BEST TUNED RANDOM FOREST
MAE  : 386.8777
RMSE : 1097.1843
R²   : 0.8820


In [6]:
# ============================================================
# EXTRA TREES REGRESSOR — HYPERPARAMETER TUNING
# ============================================================

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# ------------------------------------------------------------
# Use the same preprocessed data used for Random Forest
# ------------------------------------------------------------

# X_train_processed and X_test_processed should already exist
# from the previous preprocessing section.

print("Training Extra Trees with RandomizedSearchCV...")

# ------------------------------------------------------------
# Base model
# ------------------------------------------------------------

et_base = ExtraTreesRegressor(
    random_state=42,
    n_jobs=-1
)

# ------------------------------------------------------------
# Hyperparameter search space
# ------------------------------------------------------------

et_param_grid = {
    "n_estimators": [200, 300, 500, 700],
    "max_depth": [None, 10, 20, 30, 40],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": [0.5, 0.7, 1.0]
}

# ------------------------------------------------------------
# Randomized Search
# ------------------------------------------------------------

et_search = RandomizedSearchCV(
    estimator=et_base,
    param_distributions=et_param_grid,
    n_iter=25,
    scoring="r2",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

et_search.fit(X_train_processed, y_train)

# ------------------------------------------------------------
# Best model
# ------------------------------------------------------------

best_et = et_search.best_estimator_

print("\n" + "=" * 60)
print("BEST EXTRA TREES PARAMETERS")
print("=" * 60)

print(et_search.best_params_)

print(f"\nBest CV R²: {et_search.best_score_:.4f}")

# ------------------------------------------------------------
# Test prediction
# ------------------------------------------------------------

et_pred = best_et.predict(X_test_processed)

et_mae = mean_absolute_error(y_test, et_pred)
et_rmse = np.sqrt(mean_squared_error(y_test, et_pred))
et_r2 = r2_score(y_test, et_pred)

print("\n" + "=" * 60)
print("TUNED EXTRA TREES RESULTS")
print("=" * 60)

print(f"MAE  : {et_mae:.4f}")
print(f"RMSE : {et_rmse:.4f}")
print(f"R²   : {et_r2:.4f}")

print("\n" + "=" * 60)
print("CURRENT BEST RANDOM FOREST")
print("=" * 60)

print("MAE  : 386.8777")
print("RMSE : 1097.1843")
print("R²   : 0.8820")

print("\n" + "=" * 60)
print("EXTRA TREES vs RANDOM FOREST")
print("=" * 60)

print(f"R² Difference   : {et_r2 - 0.8820:+.4f}")
print(f"MAE Difference  : {et_mae - 386.8777:+.4f}")
print(f"RMSE Difference : {et_rmse - 1097.1843:+.4f}")

Training Extra Trees with RandomizedSearchCV...


NameError: name 'X_train_processed' is not defined

In [7]:
# ============================================================
# EXTRA TREES — FAIR HYPERPARAMETER TUNING
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ------------------------------------------------------------
# 1. Load dataset
# ------------------------------------------------------------

df_et = pd.read_csv("../data/diamonds (cleaned).csv")

target_et = "Price"

X_et = df_et.drop(columns=[target_et])
y_et = df_et[target_et]

# ------------------------------------------------------------
# 2. Train-test split FIRST
# ------------------------------------------------------------

X_train_et, X_test_et, y_train_et, y_test_et = train_test_split(
    X_et,
    y_et,
    test_size=0.20,
    random_state=42
)

# ------------------------------------------------------------
# 3. Identify feature types
# ------------------------------------------------------------

categorical_et = X_train_et.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_et = X_train_et.select_dtypes(
    exclude=["object", "category"]
).columns.tolist()

# ------------------------------------------------------------
# 4. Preprocessing
# ------------------------------------------------------------

numeric_pipeline_et = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline_et = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor_et = ColumnTransformer([
    ("numerical", numeric_pipeline_et, numerical_et),
    ("categorical", categorical_pipeline_et, categorical_et)
])

# ------------------------------------------------------------
# 5. Extra Trees
# ------------------------------------------------------------

et_model = ExtraTreesRegressor(
    random_state=42,
    n_jobs=-1
)

et_pipeline = Pipeline([
    ("preprocessor", preprocessor_et),
    ("model", et_model)
])

# ------------------------------------------------------------
# 6. Hyperparameter search
# ------------------------------------------------------------

et_params = {
    "model__n_estimators": [200, 300, 500, 700],
    "model__max_depth": [None, 10, 20, 30, 40],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": [0.5, 0.7, 1.0]
}

et_search = RandomizedSearchCV(
    et_pipeline,
    param_distributions=et_params,
    n_iter=25,
    scoring="r2",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("Training Extra Trees...")
et_search.fit(X_train_et, y_train_et)

# ------------------------------------------------------------
# 7. Best model
# ------------------------------------------------------------

best_et = et_search.best_estimator_

print("\n" + "=" * 60)
print("BEST EXTRA TREES PARAMETERS")
print("=" * 60)

print(et_search.best_params_)

print(f"\nBest CV R²: {et_search.best_score_:.4f}")

# ------------------------------------------------------------
# 8. Test evaluation
# ------------------------------------------------------------

et_pred = best_et.predict(X_test_et)

et_mae = mean_absolute_error(y_test_et, et_pred)
et_rmse = np.sqrt(mean_squared_error(y_test_et, et_pred))
et_r2 = r2_score(y_test_et, et_pred)

print("\n" + "=" * 60)
print("TUNED EXTRA TREES RESULTS")
print("=" * 60)

print(f"MAE  : {et_mae:.4f}")
print(f"RMSE : {et_rmse:.4f}")
print(f"R²   : {et_r2:.4f}")

# ------------------------------------------------------------
# 9. Comparison with current best RF
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CURRENT BEST TUNED RANDOM FOREST")
print("=" * 60)

print("MAE  : 386.8777")
print("RMSE : 1097.1843")
print("R²   : 0.8820")

print("\n" + "=" * 60)
print("EXTRA TREES vs RANDOM FOREST")
print("=" * 60)

print(f"R² Difference   : {et_r2 - 0.8820:+.4f}")
print(f"MAE Difference  : {et_mae - 386.8777:+.4f}")
print(f"RMSE Difference : {et_rmse - 1097.1843:+.4f}")

C:\Users\ishan\AppData\Local\Temp\ipykernel_2364\1608323782.py:42: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_et = X_train_et.select_dtypes(


Training Extra Trees...
Fitting 5 folds for each of 25 candidates, totalling 125 fits

BEST EXTRA TREES PARAMETERS
{'model__n_estimators': 500, 'model__min_samples_split': 2, 'model__min_samples_leaf': 4, 'model__max_features': 0.7, 'model__max_depth': 40}

Best CV R²: 0.8639

TUNED EXTRA TREES RESULTS
MAE  : 418.4975
RMSE : 1279.8714
R²   : 0.8461

CURRENT BEST TUNED RANDOM FOREST
MAE  : 386.8777
RMSE : 1097.1843
R²   : 0.8820

EXTRA TREES vs RANDOM FOREST
R² Difference   : -0.0359
MAE Difference  : +31.6198
RMSE Difference : +182.6871


In [8]:
# ============================================================
# FINAL XGBOOST HYPERPARAMETER TUNING
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor

# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------

df_xgb = pd.read_csv("../data/diamonds (cleaned).csv")

target_xgb = "Price"

X_xgb = df_xgb.drop(columns=[target_xgb])
y_xgb = df_xgb[target_xgb]

print("Dataset:", X_xgb.shape)

# ------------------------------------------------------------
# 2. TRAIN-TEST SPLIT
# ------------------------------------------------------------

X_train_xgb, X_test_xgb, y_train_xgb, y_test_xgb = train_test_split(
    X_xgb,
    y_xgb,
    test_size=0.20,
    random_state=42
)

# ------------------------------------------------------------
# 3. FEATURE TYPES
# ------------------------------------------------------------

categorical_xgb = X_train_xgb.select_dtypes(
    include=["object", "category", "str"]
).columns.tolist()

numerical_xgb = X_train_xgb.select_dtypes(
    exclude=["object", "category", "str"]
).columns.tolist()

print("Numerical features:", len(numerical_xgb))
print("Categorical features:", len(categorical_xgb))

# ------------------------------------------------------------
# 4. PREPROCESSING
# ------------------------------------------------------------

numeric_pipeline_xgb = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline_xgb = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor_xgb = ColumnTransformer([
    ("numerical", numeric_pipeline_xgb, numerical_xgb),
    ("categorical", categorical_pipeline_xgb, categorical_xgb)
])

# ------------------------------------------------------------
# 5. XGBOOST MODEL
# ------------------------------------------------------------

xgb_model = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor_xgb),
    ("model", xgb_model)
])

# ------------------------------------------------------------
# 6. FINAL SEARCH SPACE
# ------------------------------------------------------------

xgb_params = {
    "model__n_estimators": [300, 500, 700, 900, 1200],

    "model__learning_rate": [
        0.01, 0.02, 0.03, 0.05
    ],

    "model__max_depth": [
        3, 4, 5, 6, 7, 8
    ],

    "model__min_child_weight": [
        1, 3, 5, 7, 10
    ],

    "model__gamma": [
        0, 0.05, 0.1, 0.2, 0.3
    ],

    "model__subsample": [
        0.7, 0.8, 0.9, 1.0
    ],

    "model__colsample_bytree": [
        0.6, 0.7, 0.8, 0.9, 1.0
    ],

    "model__reg_alpha": [
        0, 0.01, 0.1, 0.5, 1
    ],

    "model__reg_lambda": [
        0.5, 1, 2, 5, 10
    ]
}

# ------------------------------------------------------------
# 7. RANDOMIZED SEARCH
# ------------------------------------------------------------

xgb_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=xgb_params,
    n_iter=30,
    scoring="r2",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("\nStarting FINAL XGBoost tuning...")
print("30 combinations × 5 folds = 150 model fits")

xgb_search.fit(
    X_train_xgb,
    y_train_xgb
)

# ------------------------------------------------------------
# 8. BEST MODEL
# ------------------------------------------------------------

best_xgb = xgb_search.best_estimator_

print("\n" + "=" * 65)
print("BEST XGBOOST PARAMETERS")
print("=" * 65)

for key, value in xgb_search.best_params_.items():
    print(f"{key}: {value}")

print(f"\nBest CV R²: {xgb_search.best_score_:.4f}")

# ------------------------------------------------------------
# 9. TEST SET PREDICTION
# ------------------------------------------------------------

xgb_pred = best_xgb.predict(X_test_xgb)

xgb_mae = mean_absolute_error(
    y_test_xgb,
    xgb_pred
)

xgb_rmse = np.sqrt(
    mean_squared_error(
        y_test_xgb,
        xgb_pred
    )
)

xgb_r2 = r2_score(
    y_test_xgb,
    xgb_pred
)

# ------------------------------------------------------------
# 10. FINAL RESULT
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FINAL TUNED XGBOOST RESULTS")
print("=" * 65)

print(f"MAE  : {xgb_mae:.4f}")
print(f"RMSE : {xgb_rmse:.4f}")
print(f"R²   : {xgb_r2:.4f}")

# ------------------------------------------------------------
# 11. CURRENT BEST RF COMPARISON
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("CURRENT BEST RANDOM FOREST")
print("=" * 65)

print("MAE  : 386.8777")
print("RMSE : 1097.1843")
print("R²   : 0.8820")

print("\n" + "=" * 65)
print("XGBOOST vs RANDOM FOREST")
print("=" * 65)

print(f"R² Difference   : {xgb_r2 - 0.8820:+.4f}")
print(f"MAE Difference  : {xgb_mae - 386.8777:+.4f}")
print(f"RMSE Difference : {xgb_rmse - 1097.1843:+.4f}")

Dataset: (6485, 17)
Numerical features: 7
Categorical features: 10

Starting FINAL XGBoost tuning...
30 combinations × 5 folds = 150 model fits
Fitting 5 folds for each of 30 candidates, totalling 150 fits

BEST XGBOOST PARAMETERS
model__subsample: 0.7
model__reg_lambda: 5
model__reg_alpha: 0.5
model__n_estimators: 900
model__min_child_weight: 7
model__max_depth: 6
model__learning_rate: 0.02
model__gamma: 0.05
model__colsample_bytree: 1.0

Best CV R²: 0.8618

FINAL TUNED XGBOOST RESULTS
MAE  : 434.3992
RMSE : 1332.0170
R²   : 0.8333

CURRENT BEST RANDOM FOREST
MAE  : 386.8777
RMSE : 1097.1843
R²   : 0.8820

XGBOOST vs RANDOM FOREST
R² Difference   : -0.0487
MAE Difference  : +47.5215
RMSE Difference : +234.8327


In [9]:
# ============================================================
# INNOVATIVE EXPERIMENT
# HYBRID 4Cs + RESIDUAL CORRECTION FRAMEWORK
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ------------------------------------------------------------
# 1. LOAD AND CLEAN DATA
# ------------------------------------------------------------

df_hybrid = pd.read_csv("../data/diamonds (cleaned).csv")

# Remove exact duplicate rows for a consistent clean experiment
df_hybrid = df_hybrid.drop_duplicates().reset_index(drop=True)

target = "Price"

X_all = df_hybrid.drop(columns=[target])
y_all = df_hybrid[target]

print("Dataset after exact duplicate removal:", df_hybrid.shape)

# ------------------------------------------------------------
# 2. DEFINE 4Cs AND EXTRA FEATURES
# ------------------------------------------------------------

four_cs = [
    "Carat Weight",
    "Cut",
    "Color",
    "Clarity"
]

extra_features = [
    col for col in X_all.columns
    if col not in four_cs
]

X_4c = X_all[four_cs].copy()
X_extra = X_all[extra_features].copy()

print("\n4C features:")
print(four_cs)

print("\nExtra features:")
print(extra_features)

# ------------------------------------------------------------
# 3. SAME TRAIN-TEST SPLIT FOR ALL STAGES
# ------------------------------------------------------------

(
    X4_train,
    X4_test,
    Xextra_train,
    Xextra_test,
    y_train,
    y_test
) = train_test_split(
    X_4c,
    X_extra,
    y_all,
    test_size=0.20,
    random_state=42
)

print("\nTraining rows:", len(y_train))
print("Testing rows :", len(y_test))

# ------------------------------------------------------------
# 4. PREPROCESSING FUNCTION
# ------------------------------------------------------------

def make_preprocessor(X):
    
    numerical_cols = X.select_dtypes(
        include=["number"]
    ).columns.tolist()
    
    categorical_cols = X.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ])

    preprocessor = ColumnTransformer([
        ("numerical", numeric_pipeline, numerical_cols),
        ("categorical", categorical_pipeline, categorical_cols)
    ])

    return preprocessor


# ============================================================
# STAGE 1
# 4Cs BASE MODEL
# ============================================================

base_model_template = Pipeline([
    ("preprocessor", make_preprocessor(X4_train)),
    ("model", RandomForestRegressor(
        n_estimators=300,
        min_samples_split=5,
        min_samples_leaf=4,
        max_features=0.7,
        random_state=42,
        n_jobs=-1
    ))
])

# ------------------------------------------------------------
# 5. OOF PREDICTIONS FOR TRAINING RESIDUALS
# ------------------------------------------------------------

print("\nGenerating 5-fold OOF predictions for Stage 1...")

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_base_pred = np.zeros(len(X4_train))

for fold, (train_idx, valid_idx) in enumerate(
    kf.split(X4_train), start=1
):
    
    X_fold_train = X4_train.iloc[train_idx]
    X_fold_valid = X4_train.iloc[valid_idx]
    
    y_fold_train = y_train.iloc[train_idx]

    fold_model = Pipeline([
        ("preprocessor", make_preprocessor(X_fold_train)),
        ("model", RandomForestRegressor(
            n_estimators=300,
            min_samples_split=5,
            min_samples_leaf=4,
            max_features=0.7,
            random_state=42,
            n_jobs=-1
        ))
    ])

    fold_model.fit(
        X_fold_train,
        y_fold_train
    )

    oof_base_pred[valid_idx] = fold_model.predict(
        X_fold_valid
    )

    print(f"Fold {fold}: complete")

# ------------------------------------------------------------
# 6. TRAIN FINAL BASE MODEL ON FULL TRAINING DATA
# ------------------------------------------------------------

base_model = Pipeline([
    ("preprocessor", make_preprocessor(X4_train)),
    ("model", RandomForestRegressor(
        n_estimators=300,
        min_samples_split=5,
        min_samples_leaf=4,
        max_features=0.7,
        random_state=42,
        n_jobs=-1
    ))
])

base_model.fit(
    X4_train,
    y_train
)

base_test_pred = base_model.predict(
    X4_test
)

# ------------------------------------------------------------
# 7. CALCULATE OOF RESIDUALS
# ------------------------------------------------------------

train_residual = (
    y_train.to_numpy() - oof_base_pred
)

print("\nResidual statistics:")
print(f"Mean residual   : {train_residual.mean():.4f}")
print(f"Median residual : {np.median(train_residual):.4f}")
print(f"Std residual    : {train_residual.std():.4f}")

# ============================================================
# STAGE 2
# RESIDUAL MODEL USING EXTRA ATTRIBUTES
# ============================================================

print("\nTraining Stage 2 Residual Model...")

residual_model = Pipeline([
    ("preprocessor", make_preprocessor(Xextra_train)),
    ("model", RandomForestRegressor(
        n_estimators=300,
        min_samples_split=5,
        min_samples_leaf=4,
        max_features=0.7,
        random_state=42,
        n_jobs=-1
    ))
])

residual_model.fit(
    Xextra_train,
    train_residual
)

# ------------------------------------------------------------
# 8. PREDICT RESIDUAL FOR TEST DATA
# ------------------------------------------------------------

test_residual_pred = residual_model.predict(
    Xextra_test
)

# ------------------------------------------------------------
# 9. FINAL HYBRID PREDICTION
# ------------------------------------------------------------

hybrid_pred = (
    base_test_pred + test_residual_pred
)

# ============================================================
# 10. EVALUATION FUNCTION
# ============================================================

def evaluate_model(name, actual, predicted):

    mae = mean_absolute_error(
        actual,
        predicted
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual,
            predicted
        )
    )

    r2 = r2_score(
        actual,
        predicted
    )

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R²   : {r2:.4f}")

    return mae, rmse, r2


# ------------------------------------------------------------
# 11. BASE 4C MODEL
# ------------------------------------------------------------

base_mae, base_rmse, base_r2 = evaluate_model(
    "STAGE 1 — 4Cs BASE MODEL",
    y_test,
    base_test_pred
)

# ------------------------------------------------------------
# 12. HYBRID MODEL
# ------------------------------------------------------------

hybrid_mae, hybrid_rmse, hybrid_r2 = evaluate_model(
    "STAGE 2 — HYBRID 4Cs + RESIDUAL MODEL",
    y_test,
    hybrid_pred
)

# ============================================================
# 13. COMPARE WITH OUR CURRENT BEST MODEL
# ============================================================

print("\n" + "=" * 60)
print("CURRENT BEST TUNED RANDOM FOREST")
print("=" * 60)

print("MAE  : 386.8777")
print("RMSE : 1097.1843")
print("R²   : 0.8820")

print("\n" + "=" * 60)
print("HYBRID IMPROVEMENT")
print("=" * 60)

print(
    f"R² vs 4C Base     : {hybrid_r2 - base_r2:+.4f}"
)

print(
    f"R² vs Best RF     : {hybrid_r2 - 0.8820:+.4f}"
)

print(
    f"MAE vs Best RF    : {hybrid_mae - 386.8777:+.4f}"
)

print(
    f"RMSE vs Best RF   : {hybrid_rmse - 1097.1843:+.4f}"
)

# ============================================================
# 14. SAVE RESULTS
# ============================================================

hybrid_results = pd.DataFrame({
    "Model": [
        "4Cs Base Model",
        "Hybrid 4Cs + Residual",
        "Current Best Tuned RF"
    ],
    "MAE": [
        base_mae,
        hybrid_mae,
        386.8777
    ],
    "RMSE": [
        base_rmse,
        hybrid_rmse,
        1097.1843
    ],
    "R2": [
        base_r2,
        hybrid_r2,
        0.8820
    ]
})

print("\nFINAL COMPARISON TABLE")
display(hybrid_results)

Dataset after exact duplicate removal: (6482, 18)

4C features:
['Carat Weight', 'Cut', 'Color', 'Clarity']

Extra features:
['Shape', 'Length/Width Ratio', 'Depth %', 'Table %', 'Polish', 'Symmetry', 'Girdle', 'Culet', 'Length', 'Width', 'Height', 'Type', 'Fluorescence']

Training rows: 5185
Testing rows : 1297

Generating 5-fold OOF predictions for Stage 1...
Fold 1: complete
Fold 2: complete
Fold 3: complete
Fold 4: complete
Fold 5: complete

Residual statistics:
Mean residual   : 9.6407
Median residual : -60.7357
Std residual    : 2216.5980

Training Stage 2 Residual Model...

STAGE 1 — 4Cs BASE MODEL
MAE  : 1035.5434
RMSE : 2409.2814
R²   : 0.4309

STAGE 2 — HYBRID 4Cs + RESIDUAL MODEL
MAE  : 805.4889
RMSE : 1721.9210
R²   : 0.7093

CURRENT BEST TUNED RANDOM FOREST
MAE  : 386.8777
RMSE : 1097.1843
R²   : 0.8820

HYBRID IMPROVEMENT
R² vs 4C Base     : +0.2784
R² vs Best RF     : -0.1727
MAE vs Best RF    : +418.6112
RMSE vs Best RF   : +624.7367

FINAL COMPARISON TABLE


,Model,MAE,RMSE,R2
0,4Cs Base Model,1035.543351,2409.281423,0.430907
1,Hybrid 4Cs + Residual,805.488867,1721.921047,0.709307
2,Current Best Tuned RF,386.877700,1097.184300,0.882000
